# 02_Preprocessing

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import sys
from sklearn.model_selection import train_test_split

# Calculamos la raíz del proyecto 
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# Convertimos a objeto Path para manipular rutas de forma limpia
ROOT_DIR = Path(root_path)


import src.utils.utils as utils

# Autoreload para que detecte cambios en utils.py sin reiniciar el kernel
%load_ext autoreload
%autoreload 2


DATA_PATH = ROOT_DIR / "notebooks" / "data" / "processed" / "telco_customers_churn_EDA.parquet"
REPORTS_DIR = ROOT_DIR / "reports" / "figures"

# Crear carpetas si no existen
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Caragmos el dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo en: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)

# Normalización básica 
df.columns = (
    df.columns
      .str.strip()
      .str.replace(r'(?<!^)(?=[A-Z])', '_', regex=True)
      .str.replace(r'[^\w]+', '_', regex=True)
      .str.lower()
)

df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,customer_i_d,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_t_v,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Por si acaso, volvemos a tipar las varaibles:

In [4]:
vars_num = ["tenure", "monthly_charges", "total_charges"]
var_id = ["customer_i_d"]

cols_cat = [col for col in df.columns if col not in vars_num + var_id]

df[cols_cat] = df[cols_cat].astype('category')

for col in vars_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[var_id] = df[var_id].astype('string')

# 1. Separación de varaibles: Target y Features

In [5]:
TARGET = "churn"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (7043, 20)
Shape y: (7043,)


Train test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # importante en problemas desbalanceados. Esto mantiene la proporción de las clases de churn
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (5634, 20)
Test size: (1409, 20)


Identificamos los tipos de variables que tenemos. 

In [7]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

Numéricas: ['tenure', 'monthly_charges', 'total_charges']
Categóricas: ['gender', 'senior_citizen', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_t_v', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method']


Por si acaso, en caso de que hubiera, imputamos los valores nulos mediante la media para las numéricas y por la más frecuente para las categóricas, para no tener problemas con algoritmos como SVM o Regresión Logística:

# 2. Nulos y Encoding

In [8]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="mean")
categorical_imputer = SimpleImputer(strategy="most_frequent")

Ahora, mediante One-Hot Encoding cremos una columna nueva por cada categoría:

In [9]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore", # Por si en producción aparecen variables categóricas nuevas
    sparse_output=False
)

# 3. Escalado de varaibles numéricas

Escalamos las varaibles numéricas para que todas tengan la misma "escala" y no confundir así a los modelos. Transforma cada valor siguiendo la fórmula del Z-score: $$z = \frac{x - \mu}{\sigma}$$

In [10]:
from sklearn.preprocessing import StandardScaler

numeric_scaler = StandardScaler()

# 4. Unificación

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Configuramos el transformador que separa y procesa cada tipo de columna
preprocessor = ColumnTransformer(
    transformers=[
        # Sub-proceso para variables numéricas (tenure, cargos, etc.)
        ("num", 
         Pipeline([
             ("imputer", numeric_imputer), # Rellena valores faltantes 
             ("scaler", numeric_scaler)    # Estandariza la escala 
         ]), 
         numeric_features), # Aplica a la lista de columnas numéricas

        # Sub-proceso para variables categóricas (género, contrato, etc.)
        ("cat", 
         Pipeline([
             ("imputer", categorical_imputer), # Rellena nulos con el valor más frecuente
             ("encoder", categorical_encoder)  # Convierte texto a columnas binarias (0 y 1)
         ]), 
         categorical_features) # Aplica a la lista de columnas categóricas
    ]
)

# Creamos el Pipeline que encapsula toda la lógica de limpieza
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor) # Ejecuta el transformador definido arriba como primer paso
])

In [12]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train) # aprende del train
X_test_processed = preprocessing_pipeline.transform(X_test) # aplica al test sin “mirarlo” (no data leakage)

print("Shape procesado train:", X_train_processed.shape)
print("Shape procesado test:", X_test_processed.shape)

Shape procesado train: (5634, 46)
Shape procesado test: (1409, 46)


# 5. Guardamos el pipeline

Antes de guardarlo, añadimos lo siguiente para tener interpretabilidad:

In [13]:
# Obtener nombres de variables tras OneHot
cat_features_encoded = preprocessing_pipeline.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .named_steps["encoder"] \
    .get_feature_names_out(categorical_features)

all_features = list(numeric_features) + list(cat_features_encoded)

print("Total features:", len(all_features))

Total features: 46


In [14]:
import joblib
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(preprocessing_pipeline, MODEL_DIR / "preprocessing_pipeline.pkl")

['models\\preprocessing_pipeline.pkl']